# 01 · Business questions, data check and cleaning

**Who we work for:** a travel agency that books flights from New York for its clients. The clients fly out of JFK, LGA and EWR.

## Our 3 business questions
1. **Which flights will leave more than 15 minutes late, or be cancelled, early enough to warn the clients who booked them?**
2. **Which airlines, airports and times of day are the most reliable**, so we know what to recommend when a client chooses a flight?
3. **How much of the trouble comes from the weather, and how much is a "domino effect"** from earlier late flights of the same plane?

Everything below checks the data before we trust any number, and fixes only what must be fixed. **The original files in `data/` are never changed**: the cleaned data is saved as new files in `data/processed/`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src import common

import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 170)
common.require_raw_files()

## 1. Load the five files
We read them exactly as they are. `flights` is the main table (one row per flight that left New York in 2023); the others describe the weather, the planes, the airports and the airlines.

In [2]:
flights = pd.read_csv(common.RAW / "flights.csv")
weather = pd.read_csv(common.RAW / "weather.csv")
planes = pd.read_csv(common.RAW / "planes.csv")
airports = pd.read_csv(common.RAW / "airports.csv")
airlines = pd.read_csv(common.RAW / "airlines.csv")
before = {name: hashlib.sha256((common.RAW / name).read_bytes()).hexdigest() for name in common.RAW_FILES}   # a fingerprint of every original file
assert len(flights) == common.FLIGHT_ROWS
print({"flights": len(flights), "weather": len(weather), "planes": len(planes), "airports": len(airports), "airlines": len(airlines)})
flights.head(3)

{'flights': 435352, 'weather': 26207, 'planes': 4840, 'airports': 1255, 'airlines': 14}


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2023,1,1,1.0,2038,203.0,328.0,3,205.0,UA,628,N25201,EWR,SMF,367.0,2500.0,20.0,38.0,2023-01-01 20:00:00
1,2023,1,1,18.0,2300,78.0,228.0,135,53.0,DL,393,N830DN,JFK,ATL,108.0,760.0,23.0,0.0,2023-01-01 23:00:00
2,2023,1,1,31.0,2344,47.0,500.0,426,34.0,B6,371,N807JB,JFK,BQN,190.0,1576.0,23.0,44.0,2023-01-01 23:00:00


## 2. Check every column, before changing anything
For each column: its type, how many values are missing, how many different values it has, and its smallest and largest value.

In [3]:
def profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        rows.append({"column": c, "type": str(s.dtype), "missing": int(s.isna().sum()),
                     "distinct": int(s.nunique()), "min": s.min(), "max": s.max()})
    return pd.DataFrame(rows)

print("FLIGHTS"); print(profile(flights).to_string(index=False))

FLIGHTS


        column    type  missing  distinct                 min                 max
          year   int64        0         1                2023                2023
         month   int64        0        12                   1                  12
           day   int64        0        31                   1                  31
      dep_time float64    10738      1365                 1.0              2400.0
sched_dep_time   int64        0      1060                 500                2359
     dep_delay float64    10738       946               -50.0              1813.0
      arr_time float64    11453      1433                 1.0              2400.0
sched_arr_time   int64        0      1200                   1                2359
     arr_delay float64    12534      1019               -97.0              1812.0
       carrier     str        0        14                  9E                  YX
        flight   int64        0      1972                   1                1972
       tailnum  

In [4]:
print("WEATHER"); print(profile(weather).to_string(index=False))
print("\nPLANES"); print(profile(planes).to_string(index=False))

WEATHER
    column    type  missing  distinct                 min                 max
    origin     str        0         3                 EWR                 LGA
      year   int64        0         1                2023                2023
     month   int64        0        12                   1                  12
       day   int64        0        31                   1                  31
      hour   int64        0        24                   0                  23
      temp float64        2       123                 5.0                96.0
      dewp float64        2       132               -17.0                77.0
     humid float64        2      2989               10.54               100.0
  wind_dir float64     1225        37                 0.0               360.0
wind_speed float64     1041        31                 0.0            37.97574
 wind_gust float64     1041        31                 0.0           43.701722
    precip float64        0        28                 0.

In [5]:
print("AIRPORTS"); print(profile(airports).to_string(index=False))
print("\nAIRLINES"); print(profile(airlines).to_string(index=False))

AIRPORTS
column    type  missing  distinct                       min                           max
   faa     str        0      1255                       AAF                           ZZV
  name     str        0      1252 Aberdeen Regional Airport Zephyrhills Municipal Airport
   lat float64        0      1255                 17.995278                     71.285402
   lon float64        0      1253               -176.645996                    174.113998
   alt float64        0       860                    -115.0                        9070.0
    tz float64       48         9                     -10.0                           8.0
   dst     str       48         4                         A                             U
 tzone     str      119        11              America/Adak              Pacific/Honolulu

AIRLINES
 column type  missing  distinct                  min                   max
carrier  str        0        14                   9E                    YX
   name  str        0

**What the profile already tells us**
- Missing departure times: these are the **cancelled** flights.
- Some flights have a departure time but no arrival time: they were probably **diverted**.
- `planes.speed` is 0 in every row, so that column carries no information.
- The weather table has fewer rows than 3 airports x 365 days x 24 hours, so some hours are missing.
- `time_hour` is stored as text, and the times (`dep_time`, `sched_dep_time`...) are numbers like 1530 that look like clock times but are not.

## 3. Look for problems, one by one
Each check gives a count. We keep a dictionary `found` and turn it into the data report at the end.

In [6]:
found = {}
found["cancelled"] = int(flights["dep_time"].isna().sum())
found["diverted"] = int((flights["dep_time"].notna() & flights["arr_time"].isna()).sum())
found["dep_time_2400"] = int((flights["dep_time"] == 2400).sum())
found["missing_tailnum"] = int(flights["tailnum"].isna().sum())
found["tailnum_not_in_planes"] = int((flights["tailnum"].notna() & ~flights["tailnum"].isin(planes["tailnum"])).sum())
print({k: found[k] for k in ["cancelled", "diverted", "dep_time_2400", "missing_tailnum", "tailnum_not_in_planes"]})
print("flights whose airline is not in airlines.csv:", int((~flights["carrier"].isin(airlines["carrier"])).sum()))
print("flights whose destination is not in airports.csv:", int((~flights["dest"].isin(airports["faa"])).sum()))

{'cancelled': 10738, 'diverted': 715, 'dep_time_2400': 27, 'missing_tailnum': 1913, 'tailnum_not_in_planes': 9371}
flights whose airline is not in airlines.csv: 0
flights whose destination is not in airports.csv: 0


**Plane table.** Speed is 0 everywhere and some planes have no build year.

In [7]:
assert (planes["speed"] == 0).all(), "speed is not 0 in every row"
found["planes_speed_all_zero"] = int((planes["speed"] == 0).sum())
found["planes_year_missing"] = int(planes["year"].isna().sum())
print({k: found[k] for k in ["planes_speed_all_zero", "planes_year_missing"]})

{'planes_speed_all_zero': 4840, 'planes_year_missing': 89}


**Weather table.** Two different problems: whole hours with no row at all (a flight then has no weather to join), and rows where some readings are missing.

In [8]:
weather_cols = ["temp", "dewp", "humid", "wind_dir", "wind_speed", "wind_gust", "precip", "pressure", "visib"]
assert not weather.duplicated(["origin", "time_hour"]).any()
own_hour = flights[["origin", "time_hour"]].merge(weather[["origin", "time_hour"]], on=["origin", "time_hour"], how="left", indicator=True)
assert len(own_hour) == len(flights)
found["weather_gap"] = int((own_hour["_merge"] == "left_only").sum())
found["weather_values_missing"] = int(weather[weather_cols].isna().any(axis=1).sum())
print({k: found[k] for k in ["weather_gap", "weather_values_missing"]})
print("weather rows per airport:", weather.groupby("origin").size().to_dict(), "(a full year is 8,760)")
print("missing readings:", weather[weather_cols].isna().sum()[lambda s: s > 0].to_dict())

{'weather_gap': 826, 'weather_values_missing': 1241}
weather rows per airport: {'EWR': 8736, 'JFK': 8736, 'LGA': 8735} (a full year is 8,760)
missing readings: {'temp': 2, 'dewp': 2, 'humid': 2, 'wind_dir': 1225, 'wind_speed': 1041, 'wind_gust': 1041, 'pressure': 10, 'visib': 23}


**Delays.** A few flights are extremely late. We check whether they look like errors or like real events.

In [9]:
found["extreme_delay"] = int((flights["dep_delay"] > 600).sum())
d = flights["dep_delay"]
print("flights more than 10 hours late:", found["extreme_delay"])
print("largest delay (minutes):", d.max(), "| median:", d.median(), "| mean:", round(d.mean(), 1))
print("most extreme flights (a few rows):")
flights.loc[d.nlargest(3).index, ["month", "day", "carrier", "sched_dep_time", "dep_time", "dep_delay"]]

flights more than 10 hours late: 501
largest delay (minutes): 1813.0 | median: -2.0 | mean: 13.8
most extreme flights (a few rows):


,month,day,carrier,sched_dep_time,dep_time,dep_delay
421049,12,17,AA,1340,1953.0,1813.0
331341,10,1,AA,659,1240.0,1781.0
140841,4,25,AA,659,1201.0,1742.0


**Duplicates, impossible values and the same value written in different ways.**

In [10]:
checks = {
    "exact duplicate flights": int(flights.duplicated().sum()),
    "weather rows repeated for one airport and hour": int(weather.duplicated(["origin", "time_hour"]).sum()),
    "air_time or distance at or below 0": int(((flights["air_time"] <= 0) | (flights["distance"] <= 0)).sum()),
    "scheduled hour outside 0-23 or minute outside 0-59": int((~flights["hour"].between(0, 23) | ~flights["minute"].between(0, 59)).sum()),
    "scheduled time different from hour and minute": int((flights["sched_dep_time"] != flights["hour"] * 100 + flights["minute"]).sum()),
    "humidity outside 0-100": int((~weather["humid"].dropna().between(0, 100)).sum()),
}
spelling = {c: int((flights[c] != flights[c].str.strip().str.upper()).sum()) for c in ["carrier", "origin", "dest"]}
found["duplicates_and_impossible_values"] = sum(checks.values())
found["same_value_different_spelling"] = sum(spelling.values())
print(checks)
print("codes with spaces or lower case:", spelling)

{'exact duplicate flights': 0, 'weather rows repeated for one airport and hour': 0, 'air_time or distance at or below 0': 0, 'scheduled hour outside 0-23 or minute outside 0-59': 0, 'scheduled time different from hour and minute': 0, 'humidity outside 0-100': 0}
codes with spaces or lower case: {'carrier': 0, 'origin': 0, 'dest': 0}


**Dates and times stored the wrong way.** `time_hour` is text, and the four time columns are numbers such as 1530 (meaning 15:30). Subtracting such numbers gives nonsense (1005 minus 955 is 50, but the real gap is 10 minutes), so all time calculations use real date-times.

In [11]:
found["time_hour_as_text"] = 0 if pd.api.types.is_datetime64_any_dtype(flights["time_hour"]) else len(flights)
found["times_as_hhmm_numbers"] = len(flights)
print({k: found[k] for k in ["time_hour_as_text", "times_as_hhmm_numbers"]})

{'time_hour_as_text': 435352, 'times_as_hhmm_numbers': 435352}


## 4. What this data does not tell us
- **How it was collected:** we only know what the `nycflights23` package says (see the README). We have no details on how each record was captured.
- **Period and place:** one year only (2023), and only flights that **left** JFK, LGA or EWR. Flights arriving in New York are not here.
- **Who is missing:** there is no information about passengers, tickets, prices or connections, so we cannot say how much a late flight costs a client.
- **Why flights are late:** the data has no delay cause (weather, airline, air traffic...). We can only look for patterns.
- **Weather:** it is recorded at the three New York airports only, not at the destination or along the route.
- **Times:** departure times are New York time; arrival times are the local time at the destination.

## 5. Clean: fix, remove, or keep and flag
**Rule we follow:** never throw a flight away. A cancelled flight is exactly what the agency wants to warn about, so it stays and gets a flag. Every decision is written in the data report at the end.

**Flights**
- convert `time_hour` to a real date-time;
- write midnight as 0 instead of 2400 in `dep_time`;
- add `row_id` (a number for each flight, in the original order), `sched_dep_dt` (the scheduled departure as a real date-time), the flags `cancelled`, `diverted`, `no_tailnum`;
- add the target `late_or_cancelled` (1 = cancelled, or left more than 15 minutes late);
- add `part`: `fit` (Jan-Jun), `valid` (Jul-Sep), `test` (Oct-Dec). The models never train on the test months or are tuned on them, and are evaluated on them only once, at the very end. (This notebook does print their overall late and cancelled rates, and we used those when designing the project: see D-004, D-005 and D-025 in the second brain.)

In [12]:
f = flights.copy()
f["time_hour"] = pd.to_datetime(f["time_hour"])
f["dep_time"] = f["dep_time"].replace(2400, 0)
f["row_id"] = np.arange(len(f))
f["sched_dep_dt"] = f["time_hour"] + pd.to_timedelta(f["minute"], unit="m")
f["cancelled"] = f["dep_time"].isna().astype("int8")
f["diverted"] = (f["dep_time"].notna() & f["arr_time"].isna()).astype("int8")
f["no_tailnum"] = f["tailnum"].isna().astype("int8")
f["late_or_cancelled"] = common.make_late_label(f)
f["part"] = common.split_part(f["month"])
assert len(f) == common.FLIGHT_ROWS and f["row_id"].is_unique and f["dep_time"].max() < 2400
print(f.groupby("part")[["late_or_cancelled", "cancelled"]].agg(["size", "mean"]).round(4))

      late_or_cancelled         cancelled        
                   size    mean      size    mean
part                                             
fit              222402  0.2405    222402  0.0223
test             104469  0.1406    104469  0.0021
valid            108481  0.2912    108481  0.0512


**Weather.** We build a complete table with one row for every airport and every hour of 2023. An hour that was missing gets the last reading taken at most 3 hours before it, and is flagged `weather_filled = 1`. Readings that were missing inside an existing row stay missing (gradient boosting handles gaps; the other models will get a median plus a flag in notebook 06a).

In [13]:
hours = pd.date_range("2023-01-01 00:00", "2023-12-31 23:00", freq="h")
w = weather.copy()
w["time_hour"] = pd.to_datetime(w["time_hour"])
assert w["time_hour"].between(hours[0], hours[-1]).all()
grid = pd.MultiIndex.from_product([sorted(w["origin"].unique()), hours], names=["origin", "time_hour"]).to_frame(index=False)
grid = grid.merge(w[["origin", "time_hour"] + weather_cols], on=["origin", "time_hour"], how="left")
is_new = ~grid.set_index(["origin", "time_hour"]).index.isin(w.set_index(["origin", "time_hour"]).index)
filled = grid.groupby("origin")[weather_cols].ffill(limit=3)
grid.loc[is_new, weather_cols] = filled.loc[is_new, weather_cols]
grid["weather_filled"] = is_new.astype("int8")
print("rows:", len(grid), "| rows added for missing hours:", int(is_new.sum()), "| still without readings after filling:", int(grid.loc[is_new, weather_cols].isna().all(axis=1).sum()))

rows: 26280 | rows added for missing hours: 73 | still without readings after filling: 63


**Planes.** Drop the useless `speed` column and flag planes with no build year.

In [14]:
p = planes.drop(columns="speed")
p["year_missing"] = p["year"].isna().astype("int8")
print(p.shape, "| planes without a build year:", int(p["year_missing"].sum()))

(4840, 9) | planes without a build year: 89


## 6. Save the cleaned data as new files
They go in `data/processed/`, which is never committed to GitHub.

In [15]:
common.PROC.mkdir(parents=True, exist_ok=True)
f.to_parquet(common.PROC / "flights_clean.parquet", index=False)
grid.to_parquet(common.PROC / "weather_clean.parquet", index=False)
p.to_parquet(common.PROC / "planes_clean.parquet", index=False)
after = {name: hashlib.sha256((common.RAW / name).read_bytes()).hexdigest() for name in common.RAW_FILES}
assert after == before, "an original file changed!"
common.REPORTS.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"file": list(after), "bytes": [(common.RAW / n).stat().st_size for n in after], "sha256": list(after.values())}).to_csv(common.REPORTS / "01_raw_fingerprint.csv", index=False)
print("saved:", sorted(x.name for x in common.PROC.glob("*_clean.parquet")))

saved: ['flights_clean.parquet', 'planes_clean.parquet', 'weather_clean.parquet']


## 7. Data report
One line per problem: how many rows are affected, what we did, and why.

In [16]:
decisions = {
    "cancelled": ("keep, add flag `cancelled`", "These flights never left, but they are exactly the ones the agency must warn about. They count as late in the target."),
    "diverted": ("keep, add flag `diverted`", "They left (so their delay is known) but have no arrival time. Arrival columns are never used to predict, so nothing is lost."),
    "dep_time_2400": ("fix: write 2400 as 0", "2400 is midnight written a second way; all other midnight departures use 0."),
    "missing_tailnum": ("keep, add flag `no_tailnum`", "Plane details cannot be joined. The flight stays; plane columns are simply missing for it."),
    "tailnum_not_in_planes": ("keep", "The plane table does not list every plane. Plane columns stay missing for these flights."),
    "planes_speed_all_zero": ("drop the column", "Speed is 0 in every row, so it holds no information."),
    "planes_year_missing": ("keep, add flag `year_missing`", "The plane's age cannot be computed, but its other details are still useful."),
    "weather_gap": ("complete the hourly table; fill from the last reading within 3 hours; flag `weather_filled`", "A flight with no weather row would lose its weather. Filling from a recent reading is better than dropping the flight."),
    "weather_values_missing": ("keep as missing", "A missing wind or visibility reading means no measurement. Inventing a value would hide that; models handle it explicitly later."),
    "extreme_delay": ("keep; report the median next to the mean", "Delays of 10+ hours are real overnight delays, not typing errors. They pull averages up, so we show medians too."),
    "duplicates_and_impossible_values": ("none needed", "We checked exact duplicates, repeated weather rows, air time or distance at or below 0, impossible hours and minutes, and humidity outside 0-100: none found."),
    "same_value_different_spelling": ("none needed", "Airline, origin and destination codes have no stray spaces or lower-case variants."),
    "time_hour_as_text": ("convert to a real date-time", "Needed to join the weather and to add the scheduled minute."),
    "times_as_hhmm_numbers": ("keep as given; build `sched_dep_dt` for every time calculation", "1530 means 15:30, so subtracting these numbers gives nonsense. Real date-times avoid that."),
}
assert set(decisions) == set(found), "every problem found needs a decision"
report = pd.DataFrame([{"problem": k, "rows": found[k], "action": decisions[k][0], "reason": decisions[k][1]} for k in decisions])
common.REPORTS.mkdir(parents=True, exist_ok=True)
report.to_csv(common.REPORTS / "01_data_report.csv", index=False)
print(report.to_string(index=False))

                         problem   rows                                                                                      action                                                                                                                                                        reason
                       cancelled  10738                                                                  keep, add flag `cancelled`                                         These flights never left, but they are exactly the ones the agency must warn about. They count as late in the target.
                        diverted    715                                                                   keep, add flag `diverted`                                  They left (so their delay is known) but have no arrival time. Arrival columns are never used to predict, so nothing is lost.
                   dep_time_2400     27                                                                        fix: write 2400 as 

## 8. Final check
Restart and Run All rebuilds everything above from the original files with no manual step. The three cleaned files and the report now exist, no flight was lost, and the original file is untouched.

In [17]:
print("flights kept:", len(f), "of", len(flights))
print("weather rows:", len(grid), "| planes:", len(p))
print("all 5 original files unchanged since the start of this notebook (fingerprints in reports/01_raw_fingerprint.csv):", after == before)

flights kept: 435352 of 435352
weather rows: 26280 | planes: 4840
all 5 original files unchanged since the start of this notebook (fingerprints in reports/01_raw_fingerprint.csv): True
